# Visium HD demo data: data-driven analysis buffer (paper section 2)

All public 10x Visium HD datasets processed with "v1 - Protocol 2.0" that have an H&E image
(11 datasets), at 8 µm. Included: the 8 human samples, all tissue blocks or cores whose lateral
edges were cut (trimmed surgical tissue, TMA punches). Excluded (`dataset_type = visium_demo_excluded`):
mouse brain, mouse embryo (whole-organ / whole-organism sections, natural boundaries where edge effects
and anatomy, e.g. cell-poor cortical layer 1, cannot be separated) and rat liver (lobe piece, capsule
on part of the boundary).

**Border = tissue edge.** Analysed bins: raw 8 µm bins inside the image-only tissue mask (Otsu on the
hires H&E) and inside the capture area, n_counts > 0 (holes <= 1024 µm² filled, components > 1000 bins).
Each bin's distance is to the edge of the **whole** image mask: where the capture area cuts through
tissue that continues, the cut is not a border (the effect is physical, at the tissue's own edge).
The same mask defines
"outside" for the diffusion fit (section 3).

**Border effect** = piecewise-linear elbow per component, fitted on bins within 100 µm of the tissue edge,
constrained to rise into the tissue and then plateau (m >= 0, `bosperrus.RisingPiecewiseLinearFit`),
winning against a constant by AIC, and **tested per component against interior windows of the same component**
(`bosperrus.interior_window_test`): squares of side 200 µm (2x the fit depth) at least 100 µm from the tissue
edge, no two sharing more than half their area, up to 200 per component, each fitted exactly like the tissue edge
against the distance to the window's own edge. A window has no physical border, so its elbow comes from tissue
structure. Statistic: steepness = -effect strength / half-life (the drop per µm; large only for a dip that is both
deep and narrow; 0 without a rising elbow); p = (1 + #windows at least as steep) / (1 + #windows). p > 0.05:
buffer dropped. Components where fewer than 19 windows fit (smaller than about 0.8 mm across: the smallest TMA cores and
tissue pieces) are **untested** and keep their buffer as fitted. Nothing is tuned on the tissue edges.
BLADE (as published) is the benchmark.

Supplements at the end: an 8 vs. 2 µm resolution check of the elbow, and the same panels with
Space Ranger's `in_tissue` bins. The paired Visium HD / Stereo-seq comparison (section 3) is in
`ST_paired.ipynb`.

Everything is read from `results/spatial_omics/st_reroll/`, written on a compute node by
`src/spatial_omics/st_precompute.py` (`jobs/spatial_omics_st_precompute.sbatch <sample>`, then
`<sample> --image-mask`, `--in-tissue`, `--tissue-edge --resolution {8,2}`, `--blade-image-border`; all in
`jobs/spatial_omics_window_test.sbatch`). Test status per component: `window_test_helpers.py`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bosperrus import FIT_PALETTE, plot_fit

from window_test_helpers import ALPHA, scatter_by_status, test_columns

sys.path.insert(0, "../../src/spatial_omics")

# Everything below reads precomputed per-sample results -- written on a compute
# node by src/spatial_omics/st_precompute.py (see st_precompute.sbatch), which runs
# bosperrus.load_filtered + identify_analysis_buffer_from_filtered (+ BLADE) and
# bosperrus.quantify_diffusion_from_raw at 8um for every sample in st_samples.csv.
TEXTWIDTH = (8.5 * 2.54 - 4.0) / 2.54  # manuscript text width (letterpaper, 2 cm margins), inches
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "svg.fonttype": "none"})
RESULTS_DIR = Path("../../results/spatial_omics/st_reroll")
SAMPLES = pd.read_csv("../../src/spatial_omics/st_samples.csv")

SAMPLES = SAMPLES[SAMPLES["dataset_type"] == "visium_demo"].reset_index(drop=True)
print(len(SAMPLES), "demo samples:", ", ".join(SAMPLES["name"]))

# Arial 12 throughout. Arial is not installed on the cluster: text is set in the metric-identical Liberation Sans
# (bundled in figure_plotting/fonts/, registered here so every node has it) and every SVG's font name is changed to
# Arial on saving (text stays editable, svg.fonttype = "none").
from matplotlib import font_manager

for _ttf in sorted(Path("../fonts").glob("LiberationSans-*.ttf")):
    font_manager.fontManager.addfont(str(_ttf))
assert "LiberationSans" in font_manager.findfont("Liberation Sans", fallback_to_default=False)
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Liberation Sans"],
        "font.size": 12,
        "axes.titlesize": 12,
        "axes.labelsize": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "figure.titlesize": 12,
        "svg.fonttype": "none",
        "mathtext.fontset": "custom",  # math text (10^-3, alpha, beta, um) in the same font
        "mathtext.rm": "Liberation Sans",
        "mathtext.it": "Liberation Sans:italic",
        "mathtext.bf": "Liberation Sans:bold",
        "mathtext.default": "regular",
    }
)
_savefig = plt.Figure.savefig


def _savefig_arial(self, fname, *args, **kwargs):
    _savefig(self, fname, *args, **kwargs)
    if str(fname).endswith(".svg"):
        Path(fname).write_text(Path(fname).read_text().replace("Liberation Sans", "Arial"))


plt.Figure.savefig = _savefig_arial  # plt.savefig goes through Figure.savefig

In [ ]:
DATASET_TYPE_MARKERS = {
    "visium_demo": "o",
    "visium_custom": "s",
    "stomics_custom": "^",
    "Visium HD": "s",
    "STOmics": "^",
}

# Tissue edge: % of spots in the buffer and elbow vs. BLADE (supplement)

All rising elbows (AIC vs. constant, rising-only fit) are shown: filled = p <= 0.05 against the component's own
interior windows, open = p > 0.05 (buffer dropped, so 0% of its spots are in the buffer), X = untested (too small
for interior windows; buffer kept as fitted). A component without a rising elbow is drawn at "none" and has 0% of
its bins in the buffer. BLADE peels from the analysed bins' own edge (capture-area cuts included).


In [ ]:
def _nan(v):
    return np.nan if v is None else v  # JSON null (e.g. ConstantFit won / BLADE never reached p >= 0.05)


WINDOWS = {}  # (suffix, component_label) -> steepness of every interior window of that component


def load_buffer_results(suffix):
    """Per-component results from {sample}_buffer{suffix}.json ("_imagemask": bins inside the image
    mask, distance to the tissue edge; "_intissue": Space Ranger's in_tissue bins) with the interior-window test
    (window_test_helpers.test_columns: effect_strength, half_life_um, steepness, p_value, n_windows, status,
    buffer_kept). Every component is kept; the windows' steepness goes to WINDOWS."""
    rows = []
    for sample in SAMPLES["name"]:
        buf = json.loads((RESULTS_DIR / f"{sample}_buffer{suffix}.json").read_text())
        bin_size_um = buf["resolution_um"]
        bins = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins{suffix}.parquet", columns=["components", "virtual"])
        n_bins = bins[~bins["virtual"]].groupby("components").size()
        for label, info in buf["per_component"].items():
            bsp_thresh_um = _nan(info["bsp_thresh_um"])  # piecewise_linear_b in um (None without a rising elbow)
            WINDOWS[(suffix, f"{sample}_{label}")] = np.array([w["steepness"] for w in info["windows"]], dtype=float)
            rows.append(
                {
                    "sample": sample,
                    "component_rank": int(label),
                    "component_label": f"{sample}_{label}",
                    "component_size": n_bins[int(label)],
                    "blade_thresh": _nan(info["blade_thresh"]),
                    "bsp_thresh": bsp_thresh_um / bin_size_um,
                    "bin_size_um": bin_size_um,
                    "blade_thresh_um": _nan(info["blade_thresh_um"]),
                    "bsp_thresh_um": bsp_thresh_um,
                    "pct_affected": info["pct_affected"],
                    **test_columns(info),
                }
            )
    df = pd.DataFrame(rows)
    df = df.merge(SAMPLES[["name", "dataset_type"]], left_on="sample", right_on="name")
    df["component_size_um"] = df["component_size"] * df["bin_size_um"] ** 2  # bin count -> physical area (um^2)
    df["sample_x"] = df["sample"].map({s: i for i, s in enumerate(sorted(df["sample"].unique()))})
    return df.sort_values(["sample", "component_rank"]).reset_index(drop=True)


def status_counts(df):
    """'x/y tested significant, z untested' over the components with a rising elbow."""
    s = df["status"].value_counts()
    tested = s.get("significant", 0) + s.get("not significant", 0)
    return f"{s.get('significant', 0)}/{tested} tested significant, {s.get('untested', 0)} untested"


results_df = load_buffer_results("_imagemask")
print("image mask:", status_counts(results_df))

# one colour per demo sample (same in main figure and supplement)
sample_pal = dict(zip(SAMPLES["name"], sns.color_palette("tab20", len(SAMPLES)), strict=True))


In [ ]:
def plot_blade_vs_bosperrus(ax, df, title, zero_pos=2.0, sep=5.0):
    """Panel b: bosperrus elbow vs. BLADE threshold on log-log axes. A missing
    buffer -- no border effect (elbow None/NaN) or BLADE = 0 -- is drawn at a
    stand-in position labelled "0" (zero_pos, left of/below the smallest real
    values, ~9um elbow / 8um BLADE), separated from them by dotted lines at sep."""
    d = df.assign(bsp_plot=df["bsp_thresh_um"].fillna(zero_pos), blade_plot=df["blade_thresh_um"].replace(0, zero_pos))
    scatter_by_criterion(ax, d, "bsp_plot", "blade_plot")
    max_val = np.nanmax(df[["blade_thresh_um", "bsp_thresh_um"]].values)
    ax.plot((sep, max_val), (sep, max_val), c="gray", ls="-")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axvline(sep, color="gray", ls=":", lw=0.8)
    ax.axhline(sep, color="gray", ls=":", lw=0.8)
    ticks = [zero_pos] + [t for t in [10, 100, 1000] if t <= max_val * 1.5]
    labels = ["0"] + [f"$10^{{{int(np.log10(t))}}}$" for t in ticks[1:]]
    ax.set_xticks(ticks, labels)
    ax.set_yticks(ticks, labels)
    ax.minorticks_off()
    ax.set_xlim(zero_pos / 1.6, max_val * 1.6)
    ax.set_ylim(zero_pos / 1.6, max_val * 1.6)
    ax.set_title(title)
    ax.set_ylabel("BLADE threshold ($\\mu m$)")
    ax.set_xlabel("bosperrus elbow ($\\mu m$)")


SAMPLE_ORDER = sorted(results_df["sample"].unique())  # x positions of the per-sample panels


def scatter_by_criterion(ax, df, x, y, sizes=(10, 500), linewidth=1.2):
    """Significant components filled, not significant (and no elbow) open, untested as X (same colour/size mapping)."""
    scatter_by_status(
        ax,
        df,
        linewidth=linewidth,
        x=x,
        y=y,
        style="dataset_type",
        hue="sample",
        size="component_size_um",
        markers=DATASET_TYPE_MARKERS,
        sizes=sizes,
        size_norm=(df["component_size_um"].min(), df["component_size_um"].max()),
        palette=sample_pal,
    )


def _sample_xticks(ax):
    ax.set_xticks(range(len(SAMPLE_ORDER)), SAMPLE_ORDER, rotation=30, ha="right")
    ax.set_xlim(-0.5, len(SAMPLE_ORDER) - 0.5)


def _log_axis_with_none(ax, axis, max_val, zero_pos=2.0, sep=5.0):
    """Log axis with a stand-in position labelled "none" (zero_pos) below the smallest real values."""
    ticks = [zero_pos] + [t for t in [10, 100, 1000] if t <= max_val * 1.5]
    labels = ["none"] + [f"$10^{{{int(np.log10(t))}}}$" for t in ticks[1:]]
    getattr(ax, f"set_{axis}scale")("log")
    getattr(ax, f"set_{axis}ticks")(ticks, labels)
    getattr(ax, f"set_{axis}lim")(zero_pos / 1.6, max_val * 1.6)
    (ax.axhline if axis == "y" else ax.axvline)(sep, color="gray", ls=":", lw=0.8)
    ax.minorticks_off()


def _linear_axis_with_none(ax, max_val, step=50, zero_pos=-30.0, sep=-15.0):
    """Linear y axis from 0 with a stand-in position labelled "none" (zero_pos) below it."""
    ticks = np.arange(0, max_val + step, step)
    ax.set_yticks([zero_pos, *ticks], ["none", *[f"{t:g}" for t in ticks]])
    ax.set_ylim(zero_pos * 1.4, max_val * 1.05)
    ax.axhline(sep, color="gray", ls=":", lw=0.8)


def plot_elbow(ax, df, title, zero_pos=2.0, linear=False, **scatter_kw):
    """Elbow (um) per component, grouped by sample; no border effect drawn at "none".
    linear: linear axis (none at -30 um) instead of log."""
    if linear:
        zero_pos = -30.0
    d = df.assign(bsp_plot=df["bsp_thresh_um"].fillna(zero_pos))
    scatter_by_criterion(ax, d, "sample_x", "bsp_plot", **scatter_kw)
    if linear:
        _linear_axis_with_none(ax, np.nanmax(df["bsp_thresh_um"]), zero_pos=zero_pos)
    else:
        _log_axis_with_none(ax, "y", np.nanmax(df["bsp_thresh_um"]), zero_pos)
    ax.set_ylabel("bosperrus elbow ($\\mu m$)")
    ax.set_xlabel("")
    ax.set_title(title)
    _sample_xticks(ax)


def plot_pct_affected(ax, df, title):
    scatter_by_criterion(ax, df, "sample_x", "pct_affected")
    ax.set_yscale("symlog", linthresh=1)  # log above 1%, linear below so components without effect (0%) stay visible
    ax.set_ylim(-0.1, 100)
    ax.set_ylabel("% of spots within bosperrus buffer")
    ax.set_xlabel("")
    ax.set_title(title)
    _sample_xticks(ax)


def save(save_as):
    plt.tight_layout()
    plt.savefig(f"{save_as}", bbox_inches="tight")
    plt.show()


def plot_supp(df, border, save_as):
    """Supplement: % of spots in the buffer, elbow vs. BLADE."""
    f, axs = plt.subplots(1, 2, figsize=(9, 4.5), width_ratios=[1.3, 1])
    plot_pct_affected(
        axs[0], df, f"Demo data, {border}:\nPercentage of affected spots\n(filled: p <= {ALPHA}, X: untested)"
    )
    plot_blade_vs_bosperrus(axs[1], df, f"Demo data, {border}:\nbosperrus vs. BLADE")
    save(save_as)


print(
    results_df[["component_label", "bsp_thresh_um", "effect_strength", "half_life_um", "pct_affected"]]
    .round(1)
    .to_string()
)
print(results_df.groupby("sample")[["pct_affected", "bsp_thresh_um", "blade_thresh_um"]].median().round(1))
plot_supp(results_df, "tissue edge", "ST_panels_supp_tissue_edge.svg")

# Interior windows per component

Each tested component's windows (`interior_window_test`, stored per component in the buffer JSON): how many fit,
how many have a rising elbow, and where the component's steepness falls among them.


In [ ]:
tested = results_df[results_df["status"].isin(["significant", "not significant"])]
rows = []
for _, r in tested.iterrows():
    w = WINDOWS[("_imagemask", r["component_label"])]
    rows.append(
        {
            "component": r["component_label"],
            "elbow_um": r["bsp_thresh_um"],
            "n_windows": len(w),
            "windows_with_elbow": float(np.mean(w > 0)),
            "window_steepness_p95": float(np.percentile(w, 95)),
            "steepness": r["steepness"],
            "p_value": r["p_value"],
            "status": r["status"],
        }
    )
windows_df = pd.DataFrame(rows)
all_w = np.concatenate([WINDOWS[("_imagemask", c)] for c in tested["component_label"]])
print(f"{len(all_w)} windows in {len(tested)} components; rising elbow in {np.mean(all_w > 0):.1%}")
print(windows_df.round(4).to_string(index=False))


# Diffusion outside the image mask, per demo sample

Pooled, component-free fit (ConstantFit vs. ExponentialDecayFit) on every raw 8 µm bin outside the
image-only H&E mask, zero-count bins included (`bosperrus.quantify_diffusion_from_raw`). The
cross-technology comparison of these fits is in `ST_paired.ipynb`.

In [ ]:
raw_pooled_df = pd.DataFrame(
    [json.loads((RESULTS_DIR / f"{sample}_diffusion_8um.json").read_text()) for sample in SAMPLES["name"]]
)
raw_pooled_df["1/beta"] = raw_pooled_df["decay_length_um"]
raw_pooled_df[
    [
        "sample",
        "best_fit_type",
        "alpha",
        "beta",
        "1/beta",
        "perc_counts_outside",
        "mask_tissue_fraction",
        "n_bins_total",
    ]
]

In [ ]:
# One bosperrus.plot_fit per sample: the pooled diffusion fit (ConstantFit vs.
# ExponentialDecayFit on every raw 8um bin outside the mask, zero-count bins
# included -- the fit behind panels e/f) over the per-distance-bin quantile bands and mean.
#
# The curve uses the fitted per-bin amplitude a (params["exponential_decay_a"]),
# not alpha: alpha = a / bin area is the cross-resolution-comparable number, but
# the plot shows counts per bin.
#
# Shared distance bin edges (log-spaced) across all subplots. n_counts+1 so zero-count
# bins (most bins far from the tissue) stay visible on a log axis.
all_distance, all_counts = [], []
for sample in raw_pooled_df["sample"]:
    pts = np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz")
    outside = pts["distance_um"] > 0
    all_distance.append(pts["distance_um"][outside])
    all_counts.append(pts["n_counts"][outside])
all_distance, all_counts = np.concatenate(all_distance), np.concatenate(all_counts) + 1

N_BINS = 60
x_edges = np.logspace(np.log10(all_distance.min()), np.log10(all_distance.max()), N_BINS + 1)
del all_distance, all_counts

N_COLS = 4
N_ROWS = int(np.ceil(len(raw_pooled_df) / N_COLS))
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(15, 5.3 * N_ROWS), sharex=True, sharey=True, squeeze=False)
for ax in axes.flat[len(raw_pooled_df) :]:
    ax.set_visible(False)
for ax, (_, row) in zip(axes.flat, raw_pooled_df.iterrows(), strict=False):  # spare axes hidden
    pts = np.load(RESULTS_DIR / f"{row['sample']}_diffusion_8um.npz")
    distance, n_counts = pts["distance_um"], pts["n_counts"]
    outside = distance > 0

    params = row["params"]
    if row["best_fit_type"] == "Exponential Decay Fit":
        a, b = params["exponential_decay_a"], params["exponential_decay_b"]

        def predict_fn(d, a=a, b=b):
            return a * np.exp(-b * d) + 1  # +1 to match the n_counts+1 shift below

    else:  # ConstantFit baseline won instead
        c = params["constant_c"] + 1

        def predict_fn(d, c=c):
            return np.full_like(d, c)

    plot_fit(
        ax,
        distance[outside],
        n_counts[outside] + 1,
        predict_fn,
        bins=x_edges,
        line_kwargs={"color": FIT_PALETTE.get(row["best_fit_type"], "red")},
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{row['sample']} ({row['dataset_type']})", fontsize=12)
    ax.set_xlabel("distance outside mask (um)")
    ax.set_ylabel("n_counts + 1 (raw, per 8um bin)")
    ax.set_ylim(bottom=0.8)  # log axis: n_counts + 1 >= 1
plt.tight_layout()
plt.show()

# Supplement: resolution check, panel a at 8 µm vs. 2 µm

Is the narrow tissue-edge dip a physical rim (same width in µm at both bin sizes) or bin geometry
(1-2 bins wide, i.e. narrower at 2 µm)? Panel a only, identical settings at both bin sizes
(`st_precompute.py <sample> --tissue-edge --resolution {8,2}`): every bin inside the image mask, zero-count
bins kept (at 2 µm many tissue bins have no counts), rising-only elbow vs. distance to the tissue edge within
100 µm, minimum component size scaled to the same area; no BLADE. Each component is tested against its own interior
windows at the same bin size. The image mask itself is at the hires H&E resolution
(2.1-5.2 µm per pixel), so the tissue edge is only defined to about that precision.


In [ ]:
from scipy.spatial import cKDTree

RESOLUTIONS = [8, 2]


def load_tissue_edge(res):
    """Per-component panel-a results of the --tissue-edge run at `res` um, with the interior-window test."""
    rows = []
    for sample in SAMPLES["name"]:
        buf = json.loads((RESULTS_DIR / f"{sample}_buffer_tissueedge_{res}um.json").read_text())
        bins = pd.read_parquet(
            RESULTS_DIR / f"{sample}_filtered_bins_tissueedge_{res}um.parquet",
            columns=["array_row", "array_col", "components"],
        )
        centre = bins[bins["components"] >= 0].groupby("components")[["array_row", "array_col"]].mean() * res
        for label, info in buf["per_component"].items():
            t = test_columns(info)
            rows.append(
                {
                    "sample": sample,
                    "component_rank": int(label),
                    "component_label": f"{sample}_{label}",
                    "resolution_um": res,
                    "bsp_thresh_um": _nan(info["elbow_um"]),
                    "pct_affected": info["pct_affected"],
                    **t,
                    "passes": t["status"] == "significant",
                    "dataset_type": "visium_demo",
                    "component_size_um": info["n_bins"] * res**2,
                    "centre_row_um": centre.loc[int(label), "array_row"],
                    "centre_col_um": centre.loc[int(label), "array_col"],
                }
            )
    df = pd.DataFrame(rows)
    df["sample_x"] = df["sample"].map({s: i for i, s in enumerate(SAMPLE_ORDER)})
    return df


edge_res = {res: load_tissue_edge(res) for res in RESOLUTIONS}
for res in RESOLUTIONS:
    print(f"{res} um: {status_counts(edge_res[res])}")

# match components between bin sizes by centre position (same capture area, grids nested)

a, b = edge_res[8], edge_res[2]
matched = []
for sample in SAMPLES["name"]:
    a_s, b_s = a[a["sample"] == sample], b[b["sample"] == sample]
    dist, j = cKDTree(b_s[["centre_row_um", "centre_col_um"]]).query(a_s[["centre_row_um", "centre_col_um"]])
    for (_, r8), jj, dd in zip(a_s.iterrows(), j, dist, strict=True):
        r2 = b_s.iloc[jj]
        matched.append(
            {
                "sample": sample,
                "component_label": r8["component_label"],
                "centre_offset_um": dd,
                "half_8": r8["half_life_um"],
                "half_2": r2["half_life_um"],
                "elbow_8": r8["bsp_thresh_um"],
                "elbow_2": r2["bsp_thresh_um"],
                "status_8": r8["status"],
                "status_2": r2["status"],
                "pass_8": r8["passes"],
                "pass_2": r2["passes"],
                "component_size_um": r8["component_size_um"],
            }
        )
matched = pd.DataFrame(matched)
print(
    f"matched {len(matched)} components (centre offset max {matched['centre_offset_um'].max():.0f} um); "
    f"significant at 8 um {matched['pass_8'].sum()}, at 2 um {matched['pass_2'].sum()}, "
    f"both {(matched['pass_8'] & matched['pass_2']).sum()}"
)
both = matched[(matched["status_8"] != "none") & (matched["status_2"] != "none")]  # rim width: any rising elbow
print(
    f"half-life of components with a rising elbow at both ({len(both)}): median {both['half_8'].median():.1f} um at 8 um, "
    f"{both['half_2'].median():.1f} um at 2 um (ratio 2/8 median {(both['half_2'] / both['half_8']).median():.2f})"
)
print(matched.round(1).to_string())

In [ ]:
def edge_profiles(sample, res, max_um=300.0, step_um=4.0, plateau_um=(500.0, 1000.0)):
    """Mean counts per um^2 vs. distance to the tissue edge (all components pooled), normalised to the
    mean between plateau_um[0] and plateau_um[1] (absolute if plateau_um is None)."""
    bins = pd.read_parquet(
        RESULTS_DIR / f"{sample}_filtered_bins_tissueedge_{res}um.parquet",
        columns=["n_counts", "components", "distance_to_border"],
    )
    bins = bins[bins["components"] >= 0]
    density = bins["n_counts"] / res**2
    if plateau_um is None:
        plateau = 1.0
    else:
        d = bins["distance_to_border"]
        plateau = density[(d >= plateau_um[0]) & (d <= plateau_um[1])].mean()
    edges = np.arange(0, max_um + step_um, max(step_um, res))
    step = pd.cut(bins["distance_to_border"], edges, right=False)
    prof = density.groupby(step, observed=True).mean() / plateau
    return np.array([iv.mid for iv in prof.index]), prof.to_numpy()


f, axs = plt.subplots(1, 4, figsize=(20, 4.5), width_ratios=[1.3, 1.3, 1, 1.3])
for ax, res in zip(axs[:2], RESOLUTIONS, strict=True):
    plot_elbow(
        ax,
        edge_res[res],
        f"tissue edge, {res} $\\mu m$ bins: bosperrus elbow\n"
        f"(filled: p <= {ALPHA}, open: p > {ALPHA}, X: untested)",
    )
ax = axs[2]
m = matched.dropna(subset=["half_8", "half_2"])
for ok, fill in [(True, True), (False, False)]:
    mm = m[(m["pass_8"] & m["pass_2"]) == ok]
    ax.scatter(
        mm["half_8"],
        mm["half_2"],
        s=np.sqrt(mm["component_size_um"]) / 30,
        c=[sample_pal[s] for s in mm["sample"]] if fill else "none",
        edgecolors=[sample_pal[s] for s in mm["sample"]],
        linewidths=1.2,
    )
lim = (1, max(m["half_8"].max(), m["half_2"].max()) * 1.5)
ax.plot(lim, lim, color="gray")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("half-life at 8 $\\mu m$ ($\\mu m$)")
ax.set_ylabel("half-life at 2 $\\mu m$ ($\\mu m$)")
ax.set_title("same component, both bin sizes\n(filled: p <= 0.05 at both)")
ax = axs[3]
for sample in SAMPLES["name"]:
    for res, ls in [(8, "--"), (2, "-")]:
        x, y = edge_profiles(sample, res)
        ax.plot(x, y, color=sample_pal[sample], ls=ls, lw=1)
ax.axhline(1, color="gray", lw=0.8)
ax.set_xlim(0, 300)
ax.set_ylim(bottom=0)
ax.set_xlabel("distance to tissue edge ($\\mu m$)")
ax.set_ylabel("counts per $\\mu m^2$ / plateau (0.5-1 mm)")
ax.set_title("edge profiles, all components pooled\n(solid: 2 $\\mu m$, dashed: 8 $\\mu m$)")
save("ST_resolution_check.svg")

# Supplement: the supplement panels with Space Ranger's `in_tissue` bins as tissue footprint

Same analysis as the main figure (fit within 100 µm of the edge, same per-component interior-window test), but the bins are
Space Ranger's filtered (`in_tissue`) bins (`bosperrus.load_filtered`) and the border is the edge of their
footprint, so capture-area cuts count as border (`st_precompute.py <sample> --in-tissue`,
`*_buffer_intissue.json`).

In [ ]:
spaceranger_df = load_buffer_results("_intissue")
print("in_tissue:", status_counts(spaceranger_df))
print(
    spaceranger_df[["component_label", "bsp_thresh_um", "effect_strength", "half_life_um", "pct_affected"]]
    .round(1)
    .to_string()
)
print(spaceranger_df.groupby("sample")[["pct_affected", "bsp_thresh_um", "blade_thresh_um"]].median().round(1))
plot_supp(spaceranger_df, "Space Ranger in_tissue", "ST_panels_supp_spaceranger.svg")

# Main figure (section 2)

Example: breast_cancer (A-C, E, F). A: H&E with the image-derived tissue mask (pink), Space Ranger's `in_tissue`
footprint (green, dotted) and the capture area (black, dashed). B: counts per 8 µm bin inside the image mask. C: the
bosperrus fit of its main component over the whole distance range (counts vs. distance to the tissue edge; 5-95% and
25-75% bands, mean; rising piecewise-linear fit and its elbow). Over the whole section the single elbow follows the
tissue-scale gradient (~300 µm) rather than the narrow dip at the edge; hence the fit within 100 µm of the edge and the
test against interior windows (E). D: the fit the analysis uses: the same component within 100 µm of the edge (the fit depth;
counts per 8 µm bin, bands and mean as in C) with its rising piecewise-linear fit and elbow. E: the
interior-window test of that component: effect strength vs. half-life of its tissue edge and of its interior windows
with a rising elbow; shaded: at least as steep as the edge (steepness = -effect strength / max(half-life, 5 µm)). F: 1
mm window on breast_cancer's edge (box in A, B) with the spots each method discards, coloured as in the MICrONS figure:
both (blue), BLADE only (green; BLADE as published, peeled from the `in_tissue` footprint), bosperrus only (turquoise;
closer to the image tissue edge than the elbow). G: steepness of every component with a rising elbow (coloured; filled
p <= 0.05, open p > 0.05, X untested) and of its own interior windows (grey; windows without a rising elbow at
"none"). H: bosperrus elbow per component (same markers; components without a rising elbow at "none").

The BLADE comparison numbers (per sample: Jaccard index and % of `in_tissue` spots discarded by each method) are
printed below for the text / supplement (`st_precompute.py <sample> --blade-image-border`).

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from bosperrus import PiecewiseLinearFit
from bosperrus.anndata_api import fit_border_profile
from scipy import ndimage
from scipy.spatial import cKDTree

FIG_SAMPLE, FIG_COMPONENT = "breast_cancer", 0  # example section (panels A, B, D, G) and its tested component
ZOOM_HALF_UM = 500  # panel G: 1 mm window on the edge of FIG_COMPONENT
IMAGE_MASK_COLOR, SPOT_MASK_COLOR = "#FF1493", "limegreen"
# buffer classes as in the MICrONS figure (BLADE = the published, "original" buffer)
PALETTE = json.loads(Path("../../fit_palette.json").read_text())
BOTH_COLOR, BLADE_ONLY_COLOR, BSP_ONLY_COLOR = (
    PALETTE["Piecewise Linear Fit"],
    PALETTE["Manual buffer"],
    PALETTE["BOSPERRUS-only buffer"],
)
COUNTS_CMAP = "YlOrRd"  # as the degree maps of the meristem / MICrONS figures
MIN_HALF_LIFE_UM = 5.0  # half-life floor of the steepness statistic (st_precompute.MIN_HALF_LIFE_UM)


def panel_titles(lettered):
    """Meristem/MICrONS style: titles left-aligned, panel letter in front ("A  title")."""
    for ax, letter in lettered:
        t = ax.get_title()
        ax.set_title("")
        ax.set_title(f"{letter}  {t}", loc="left")


def _pad_linear(a):
    """Pad a 2D coordinate array by one cell on every side, extrapolating linearly."""
    a = np.pad(a, 1, mode="edge").astype(float)
    a[0, :], a[-1, :] = 2 * a[1, :] - a[2, :], 2 * a[-2, :] - a[-3, :]
    a[:, 0], a[:, -1] = 2 * a[:, 1] - a[:, 2], 2 * a[:, -2] - a[:, -3]
    return a


def load_geometry(sample):
    """Image, mask and the (1-cell padded) grid -> mask coordinate maps of one sample."""
    raw = dict(np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz"))
    shape = (raw["array_row"].max() + 1, raw["array_col"].max() + 1)
    grid_row, grid_col = np.full(shape, np.nan), np.full(shape, np.nan)
    grid_row[raw["array_row"], raw["array_col"]] = raw["mask_row"]
    grid_col[raw["array_row"], raw["array_col"]] = raw["mask_col"]
    raw["grid_row"], raw["grid_col"] = _pad_linear(grid_row), _pad_linear(grid_col)
    raw["shape"] = shape
    return raw


def level_capture_area(geo):
    """Rotate image, mask and grid about the mask centre so the capture area's top edge is horizontal (the
    capture area sits ~0.4 deg off the image axes). Returns the rotated geometry and a function mapping mask
    coordinates (x = col, y = row) into the rotated frame."""
    gr, gc = geo["grid_row"][1:-1, 1:-1], geo["grid_col"][1:-1, 1:-1]
    phi = -np.arctan2(gr[0, 0] - gr[0, -1], gc[0, 0] - gc[0, -1])  # top edge, first -> last column
    cos, sin = np.cos(phi), np.sin(phi)
    cy, cx = (np.array(geo["mask"].shape) - 1) / 2

    def to_level(x, y):
        x, y = np.asarray(x, float) - cx, np.asarray(y, float) - cy
        return cx + cos * x - sin * y, cy + sin * x + cos * y

    def rotate(a, cval, order, centre_yx):
        # output (y, x) samples the input at centre + R(-phi) (out - centre), in (row, col) order
        mat = np.array([[cos, -sin], [sin, cos]])
        offset = centre_yx - mat @ centre_yx
        if a.ndim == 2:
            return ndimage.affine_transform(a, mat, offset, order=order, cval=cval)
        return np.stack([ndimage.affine_transform(a[..., k], mat, offset, order=order, cval=cval)
                         for k in range(a.shape[-1])], axis=-1)

    out = dict(geo)
    f = int(geo["image_factor"])
    out["mask"] = rotate(geo["mask"].astype(np.uint8), 0, 0, np.array([cy, cx])).astype(bool)
    centre_img = (np.array([cy, cx]) + 0.5) / f - 0.5  # mask centre in image px (draw_image's extent)
    out["image"] = rotate(geo["image"], 255, 1, centre_img)
    out["grid_col"], out["grid_row"] = to_level(geo["grid_col"], geo["grid_row"])
    return out, to_level


def grid_raster(geo, array_row, array_col, values):
    out = np.zeros((geo["shape"][0] + 2, geo["shape"][1] + 2))
    out[np.asarray(array_row) + 1, np.asarray(array_col) + 1] = values
    return out


def mask_slices(box, pad=2):
    """Row/column slices of the mask (mask px) covering box (x0, x1, y0, y1) plus pad pixels."""
    x0, x1, y0, y1 = box
    return slice(max(int(np.floor(y0)) - pad, 0), int(np.ceil(y1)) + pad), slice(
        max(int(np.floor(x0)) - pad, 0), int(np.ceil(x1)) + pad
    )


def grid_slices(geo, box, pad=2):
    """Row/column slices of the padded bin grid whose mask coordinates fall in box, plus pad bins."""
    x0, x1, y0, y1 = box
    gr, gc = geo["grid_row"], geo["grid_col"]
    inside = (gc >= x0) & (gc <= x1) & (gr >= y0) & (gr <= y1)
    rows, cols = np.flatnonzero(inside.any(axis=1)), np.flatnonzero(inside.any(axis=0))
    return slice(max(rows.min() - pad, 0), rows.max() + pad + 1), slice(max(cols.min() - pad, 0), cols.max() + pad + 1)


def draw_image(ax, geo, box):
    """The display image cropped to box (the SVG then holds only what is visible, no clip path needed)."""
    f = int(geo["image_factor"])
    x0, x1, y0, y1 = box
    r0, r1 = max(int(np.floor(y0 / f)) - 1, 0), int(np.ceil(y1 / f)) + 1
    c0, c1 = max(int(np.floor(x0 / f)) - 1, 0), int(np.ceil(x1 / f)) + 1
    img = geo["image"][r0:r1, c0:c1]
    ax.imshow(img, extent=(c0 * f - 0.5, (c0 + img.shape[1]) * f - 0.5, (r0 + img.shape[0]) * f - 0.5, r0 * f - 0.5),
              rasterized=True)


def draw_grid(ax, geo, box, values, kind="pcolormesh", **kw):
    """pcolormesh / contourf / contour of a padded-grid array, cropped to box."""
    rs, cs = grid_slices(geo, box)
    return getattr(ax, kind)(geo["grid_col"][rs, cs], geo["grid_row"][rs, cs], values[rs, cs], **kw)


def capture_corners(geo):
    """Capture-area corners in mask px (x, y): the outer bin centres of the grid, closed."""
    gr, gc = geo["grid_row"][1:-1, 1:-1], geo["grid_col"][1:-1, 1:-1]
    return [(gc[i, j], gr[i, j]) for i, j in [(0, 0), (0, -1), (-1, -1), (-1, 0)]]


def draw_outlines(ax, geo, footprint, lw, box, spot=True, capture=True):
    """Image mask (pink) and in_tissue footprint (green dotted), cropped to box; the capture area as one closed
    dashed rectangle (box must contain it)."""
    mask = geo["mask"]
    rs, cs = mask_slices(box)
    ax.contour(
        np.arange(mask.shape[1])[cs],
        np.arange(mask.shape[0])[rs],
        mask[rs, cs].astype(float),
        levels=[0.5],
        colors=[IMAGE_MASK_COLOR],
        linewidths=lw,
        rasterized=True,
    )
    if spot:
        draw_grid(ax, geo, box, grid_raster(geo, footprint["array_row"], footprint["array_col"], 1.0), "contour",
                  levels=[0.5], colors=[SPOT_MASK_COLOR], linewidths=lw * 1.2, linestyles=":", rasterized=True)
    if capture:
        ax.add_patch(plt.Polygon(capture_corners(geo), closed=True, fill=False, edgecolor="black", ls="--",
                                 lw=lw * 0.8))


def map_axes(ax, box, um_per_px, bar_um):
    """Crop to box (x0, x1, y0, y1, mask px), no ticks or frame, scale bar bottom right."""
    x0, x1, y0, y1 = box
    ax.set_xlim(x0, x1)
    ax.set_ylim(y1, y0)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines[:].set_visible(False)
    px = bar_um / um_per_px
    xe, ye = x1 - 0.04 * (x1 - x0), y1 - 0.05 * (y1 - y0)
    ax.plot([xe - px, xe], [ye, ye], color="black", lw=1.5)
    label = f"{bar_um / 1000:g} mm" if bar_um >= 1000 else f"{bar_um} µm"
    ax.text(xe - px / 2, ye - 0.02 * (y1 - y0), label, ha="center", va="bottom", fontsize=12,
            bbox=dict(facecolor="white", alpha=0.7, lw=0, pad=1))


geo, to_level = level_capture_area(load_geometry(FIG_SAMPLE))
UM_PER_PX = float(geo["mask_pixel_size_um"])
footprint = pd.read_parquet(
    RESULTS_DIR / f"{FIG_SAMPLE}_filtered_bins_intissue.parquet",
    columns=["array_row", "array_col", "components", "n_counts", "virtual"],
)
footprint = footprint[(footprint["components"] >= 0) & ~footprint["virtual"]]
discard = pd.read_parquet(RESULTS_DIR / f"{FIG_SAMPLE}_bins_blade_imageborder.parquet")
analysed = pd.read_parquet(
    RESULTS_DIR / f"{FIG_SAMPLE}_filtered_bins_imagemask.parquet",
    columns=["array_row", "array_col", "n_counts", "components", "virtual", "distance_to_border", "mask_row",
             "mask_col"],
)
analysed = analysed[(analysed["components"] >= 0) & ~analysed["virtual"]]
analysed["mask_col"], analysed["mask_row"] = to_level(analysed["mask_col"], analysed["mask_row"])
# A/B: the analysed bins and the whole capture area, plus 4%
cap = np.array(capture_corners(geo))
r0, r1 = min(analysed["mask_row"].min(), cap[:, 1].min()), max(analysed["mask_row"].max(), cap[:, 1].max())
c0, c1 = min(analysed["mask_col"].min(), cap[:, 0].min()), max(analysed["mask_col"].max(), cap[:, 0].max())
m = 0.04 * max(r1 - r0, c1 - c0)
SECTION_BOX = (c0 - m, c1 + m, r0 - m, r1 + m)
# G: window on a real tissue edge of FIG_COMPONENT: its edge bins more than 600 um from the capture-area border
# (capture-area cuts are not a tissue edge), the stretch at the largest array rows, centred on its median position
comp = analysed[analysed["components"] == FIG_COMPONENT]
edge_bins = comp[comp["distance_to_border"] < 8]
n_rows, n_cols = geo["shape"]
to_capture = 8 * np.minimum.reduce(
    [edge_bins["array_row"], n_rows - 1 - edge_bins["array_row"], edge_bins["array_col"], n_cols - 1 - edge_bins["array_col"]]
)
real = edge_bins[to_capture > 600]
stretch = real[real["array_row"] > real["array_row"].quantile(0.8)]
_, i = cKDTree(edge_bins[["mask_row", "mask_col"]]).query(stretch[["mask_row", "mask_col"]].median().to_numpy())
zr, zc = edge_bins.iloc[i][["mask_row", "mask_col"]]
h = ZOOM_HALF_UM / UM_PER_PX
zr -= 0.3 * h * np.sign(zr - comp["mask_row"].mean())  # shift the window a bit into the tissue
ZOOM = (zc - h, zc + h, zr - h, zr + h)

fig_buf = json.loads((RESULTS_DIR / f"{FIG_SAMPLE}_buffer_imagemask.json").read_text())
fig_info = fig_buf["per_component"][str(FIG_COMPONENT)]
print(
    f"example: {FIG_SAMPLE}_{FIG_COMPONENT}: elbow {fig_info['elbow_um']:.1f} um, effect "
    f"{fig_info['observed_effect_strength']:.3f}, half-life {fig_info['half_life_um']:.1f} um, steepness "
    f"{fig_info['steepness']:.4f}, p = {fig_info['p_value']:.3g} ({fig_info['n_windows']} windows)"
)


def load_blade_image_border():
    """Per sample: Jaccard of the in_tissue spots discarded by BLADE and by bosperrus (image border)."""
    rows = []
    for sample in SAMPLES["name"]:
        j = json.loads((RESULTS_DIR / f"{sample}_blade_imageborder.json").read_text())
        rows.append(
            {
                "sample": sample,
                "jaccard_discarded": j["jaccard_discarded"],
                "n_in_tissue": j["n_in_tissue"],
                "pct_discard_blade": 100 * j["n_discard_blade"] / j["n_in_tissue"],
                "pct_discard_bosperrus": 100 * j["n_discard_bosperrus"] / j["n_in_tissue"],
            }
        )
    return pd.DataFrame(rows)


jaccard_df = load_blade_image_border()
print(jaccard_df.round(3).to_string(index=False))

with plt.rc_context({}):
    fig = plt.figure(figsize=(9, 10.5), layout="constrained")
    fig.get_layout_engine().set(w_pad=0.02, h_pad=0.02, wspace=0.03, hspace=0.04)
    # rows 1-2 (except D): the example section; D, G, H: all samples
    axd = fig.subplot_mosaic([["A", "B", "C"], ["D", "E", "F"], ["G", "G", "H"]])
    ax_he, ax_cnt, ax_fit = axd["A"], axd["B"], axd["C"]
    ax_prof, ax_win, ax_disc = axd["D"], axd["E"], axd["F"]
    ax_steep, ax_elbow = axd["G"], axd["H"]
    zoom_rect = dict(fill=False, color="black", lw=0.8)

    # A: H&E with the masks
    draw_image(ax_he, geo, SECTION_BOX)
    draw_outlines(ax_he, geo, footprint, 0.5, SECTION_BOX)
    map_axes(ax_he, SECTION_BOX, UM_PER_PX, 1000)
    ax_he.add_patch(plt.Rectangle((ZOOM[0], ZOOM[2]), 2 * h, 2 * h, **zoom_rect))
    ax_he.set_title("H&E")

    # B: counts of the analysed bins (inside the image mask)
    counts = grid_raster(geo, analysed["array_row"], analysed["array_col"], analysed["n_counts"])
    counts[counts == 0] = np.nan
    mesh = draw_grid(
        ax_cnt,
        geo,
        SECTION_BOX,
        np.ma.masked_invalid(counts),
        cmap=COUNTS_CMAP,
        vmin=0,
        vmax=np.nanpercentile(counts, 99),
        shading="nearest",
        rasterized=True,
    )
    draw_outlines(ax_cnt, geo, footprint, 0.5, SECTION_BOX, spot=False)
    map_axes(ax_cnt, SECTION_BOX, UM_PER_PX, 1000)
    ax_cnt.add_patch(plt.Rectangle((ZOOM[0], ZOOM[2]), 2 * h, 2 * h, **zoom_rect))
    cb = plt.colorbar(mesh, cax=ax_cnt.inset_axes([1.02, 0.15, 0.04, 0.7]), extend="max")
    cb.ax.tick_params(labelsize=12, length=2, pad=1)
    cb.set_label("counts", fontsize=12, labelpad=1)
    ax_cnt.set_title("counts")

    # C: the bosperrus fit of the example component over the whole distance range (one quick illustrative fit)
    global_fit = fit_border_profile(comp["distance_to_border"].to_numpy(), comp["n_counts"].to_numpy(), rising_only=True)
    print(
        f"global fit ({FIG_SAMPLE}_{FIG_COMPONENT}, whole distance range): elbow {global_fit['elbow_um']:.0f} um, "
        f"half-life {global_fit['half_life_um']:.0f} um, effect {global_fit['observed_effect_strength']:.2f}"
    )
    d_max = comp["distance_to_border"].max()

    def piecewise(p):
        return lambda d: PiecewiseLinearFit.piecewise_plateau(
            np.asarray(d, float), p["piecewise_linear_b"], p["piecewise_linear_m"], p["piecewise_linear_c"]
        )

    plot_fit(
        ax_fit,
        comp["distance_to_border"],
        comp["n_counts"],
        piecewise(global_fit["params"]),
        bins=np.r_[np.arange(0, 200, 8), np.arange(200, d_max + 40, 40)],
        d_grid=np.linspace(0, d_max, 400),
        line_kwargs={"color": BOTH_COLOR, "linewidth": 1.5},
    )
    knot = global_fit["elbow_um"]
    ax_fit.axvspan(0, knot, color=BOTH_COLOR, alpha=0.12, lw=0)
    ax_fit.axvline(knot, color=BOTH_COLOR, ls="--", lw=1)
    ax_fit.text(knot, 0.97, f" elbow {knot:.0f} µm", transform=ax_fit.get_xaxis_transform(), va="top", fontsize=12,
                color=BOTH_COLOR)
    ax_fit.set_xlim(0, d_max)
    ax_fit.set_ylim(bottom=0)
    ax_fit.set_xlabel("distance to tissue edge (µm)")
    ax_fit.set_ylabel("counts per bin")
    ax_fit.set_title("global fit")

    # D: the analysis fit of the example component (rising piecewise linear within the fit depth), as C
    fit_depth = fig_buf["fit_max_distance_um"]  # 100 um
    near = comp[comp["distance_to_border"] <= fit_depth]
    plot_fit(
        ax_prof,
        near["distance_to_border"],
        near["n_counts"],
        piecewise(fig_info["params"]),
        bins=np.arange(0, fit_depth + 8, 8),
        d_grid=np.linspace(0, fit_depth, 200),
        line_kwargs={"color": BOTH_COLOR, "linewidth": 1.5},
    )
    knot = fig_info["elbow_um"]
    ax_prof.axvspan(0, knot, color=BOTH_COLOR, alpha=0.12, lw=0)
    ax_prof.axvline(knot, color=BOTH_COLOR, ls="--", lw=1)
    ax_prof.text(knot, 0.97, f" elbow {knot:.0f} µm", transform=ax_prof.get_xaxis_transform(), va="top", fontsize=12,
                 color=BOTH_COLOR)
    ax_prof.set_xlim(0, fit_depth)
    ax_prof.set_ylim(ax_fit.get_ylim())  # same count scale as C
    ax_prof.set_xlabel("distance to tissue edge (µm)")
    ax_prof.set_ylabel("counts per bin")
    ax_prof.set_title("edge fit")

    # E: effect strength vs. half-life of the example's tissue edge and its own interior windows
    win = pd.DataFrame(fig_info["windows"]).dropna(subset=["half_life_um"])
    steep = fig_info["steepness"]
    hl_lim = (3, 80)
    ax_win.scatter(win["half_life_um"], win["effect_strength"], s=6, c="lightgray", lw=0, rasterized=True)
    hl = np.geomspace(*hl_lim, 200)
    iso = -steep * np.maximum(hl, MIN_HALF_LIFE_UM)  # windows at least as steep as the edge lie below
    y_low = 1.3 * min(fig_info["observed_effect_strength"], win["effect_strength"].min())
    ax_win.fill_between(hl, y_low, np.maximum(iso, y_low), color=BOTH_COLOR, alpha=0.08, lw=0, zorder=0)
    ax_win.plot(hl, iso, color=BOTH_COLOR, lw=0.6, ls=":")
    ax_win.scatter(fig_info["half_life_um"], fig_info["observed_effect_strength"], s=40,
                   color=sample_pal[FIG_SAMPLE], edgecolor="black", lw=0.5, zorder=3)
    ax_win.annotate("tissue edge", (fig_info["half_life_um"], fig_info["observed_effect_strength"]),
                    xytext=(6, -2), textcoords="offset points", fontsize=12, va="top")
    ax_win.set_xscale("log")
    ax_win.set_xlim(hl_lim)
    ax_win.set_xticks([5, 10, 20, 50], ["5", "10", "20", "50"])
    ax_win.minorticks_off()
    ax_win.set_ylim(y_low, 0)
    ax_win.set_xlabel("half-life (µm)")
    ax_win.set_ylabel("effect strength")
    ax_win.set_title(f"window test (p = {fig_info['p_value']:.2g})")

    # G: steepness of each component vs. its own interior windows; a sample's tested components share its column
    ZERO_STEEP, SEP_STEEP = 6e-5, 1.1e-4  # stand-in for steepness 0 (no rising elbow), below the smallest window
    edge = results_df[results_df["status"] != "none"].copy()
    tested = edge["status"] != "untested"
    rank = edge[tested].groupby("sample").cumcount()
    n_tested = edge[tested].groupby("sample")["component_rank"].transform("size")
    spacing = np.minimum(0.35, 0.8 / np.maximum(n_tested - 1, 1))
    edge["c_x"] = edge["sample_x"].astype(float)
    edge.loc[rank.index, "c_x"] += (rank - (n_tested - 1) / 2) * spacing
    edge["jitter"] = 0.12
    edge.loc[rank.index, "jitter"] = np.minimum(0.12, spacing * 0.35)
    rng = np.random.default_rng(0)  # jitter of windows and untested components
    edge.loc[~tested, "c_x"] += rng.uniform(-0.35, 0.35, (~tested).sum())
    for _, r in edge[tested].iterrows():
        w = WINDOWS[("_imagemask", r["component_label"])]
        ax_steep.scatter(
            r["c_x"] + rng.uniform(-r["jitter"], r["jitter"], len(w)),
            np.where(w > 0, w, ZERO_STEEP),
            s=3,
            c="lightgray",
            lw=0,
            rasterized=True,
            zorder=0,
        )
    scatter_by_status(
        ax_steep,
        edge.assign(steep_plot=edge["steepness"].clip(lower=ZERO_STEEP)),
        x="c_x",
        y="steep_plot",
        hue="sample",
        style="dataset_type",
        markers=DATASET_TYPE_MARKERS,
        size="component_size_um",
        sizes=(15, 220),
        size_norm=(results_df["component_size_um"].min(), results_df["component_size_um"].max()),
        palette=sample_pal,
    )
    ax_steep.set_yscale("log")
    ax_steep.set_yticks([ZERO_STEEP, 1e-3, 1e-2, 1e-1], ["none", "$10^{-3}$", "$10^{-2}$", "$10^{-1}$"])
    ax_steep.minorticks_off()
    ax_steep.set_ylim(ZERO_STEEP / 1.6, max(edge["steepness"].max(), 0.1) * 1.3)
    ax_steep.axhline(SEP_STEEP, color="gray", ls=":", lw=0.8)
    ax_steep.set_xticks(range(len(SAMPLE_ORDER)), SAMPLE_ORDER, rotation=30, ha="right")
    ax_steep.set_xlim(-0.5, len(SAMPLE_ORDER) - 0.5)
    ax_steep.set_xlabel("")
    ax_steep.set_ylabel("steepness (µm$^{-1}$)")
    ax_steep.set_title("steepness: edges vs. own windows")
    print(status_counts(results_df))

    # H: elbows
    plot_elbow(ax_elbow, results_df, "elbows", linear=True, sizes=(15, 220), linewidth=0.8)
    ax_elbow.set_xticks(range(len(SAMPLE_ORDER)), [])
    ax_elbow.set_ylabel("elbow (µm)")

    # F: discarded spots in the zoom
    fp_counts = grid_raster(geo, footprint["array_row"], footprint["array_col"], footprint["n_counts"])
    fp_counts[fp_counts == 0] = np.nan
    draw_grid(
        ax_disc,
        geo,
        ZOOM,
        np.ma.masked_invalid(fp_counts),
        cmap=COUNTS_CMAP,
        vmin=0,
        vmax=np.nanpercentile(counts, 99),  # same scale as B
        shading="nearest",
        rasterized=True,
    )
    blade, bsp = discard["in_blade_buffer"], discard["bosperrus_discard"]
    for sel, color in ((blade & bsp, BOTH_COLOR), (blade & ~bsp, BLADE_ONLY_COLOR), (~blade & bsp, BSP_ONLY_COLOR)):
        grid = grid_raster(geo, discard.loc[sel, "array_row"], discard.loc[sel, "array_col"], 1.0)
        draw_grid(ax_disc, geo, ZOOM, grid, "contourf", levels=[0.5, 1.5], colors=[color], alpha=0.85)
    draw_outlines(ax_disc, geo, footprint, 1.0, ZOOM, capture=False)  # window > 500 um from the capture border
    map_axes(ax_disc, ZOOM, UM_PER_PX, 200)
    ax_disc.set_title("discarded spots")

    handles = [Line2D([0], [0], marker="o", ls="", color=sample_pal[s], label=s, markersize=6) for s in SAMPLE_ORDER]
    handles += [
        Line2D([0], [0], marker="o", ls="", color="dimgray", markersize=6, label=f"p ≤ {ALPHA}"),
        Line2D([0], [0], marker="o", ls="", markerfacecolor="none", markeredgecolor="dimgray", markersize=6,
               label=f"p > {ALPHA} (buffer dropped)"),
        Line2D([0], [0], marker="X", ls="", color="dimgray", markersize=6, label="untested (< ~0.8 mm)"),
        Line2D([0], [0], color=IMAGE_MASK_COLOR, lw=1, label="image-derived mask"),
        Line2D([0], [0], color=SPOT_MASK_COLOR, lw=1.2, ls=":", label="spot-derived mask (in_tissue)"),
        Line2D([0], [0], color="black", lw=0.8, ls="--", label="capture area"),
        Patch(color=BOTH_COLOR, alpha=0.85, label="discarded by both"),
        Patch(color=BLADE_ONLY_COLOR, alpha=0.85, label="discarded by BLADE only"),
        Patch(color=BSP_ONLY_COLOR, alpha=0.85, label="discarded by bosperrus only"),
    ]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False, columnspacing=1.0,
               handletextpad=0.3)
    panel_titles(zip([ax_he, ax_cnt, ax_fit, ax_prof, ax_win, ax_disc, ax_steep, ax_elbow], "ABCDEFGH", strict=True))
    plt.savefig("ST_figure_demo.svg", bbox_inches="tight", dpi=300)
    plt.show()

# Supplement: global fits of the demo samples

Per demo sample, its largest component (for breast_cancer_tma and tissue_array: the largest core): counts per 8 µm
bin vs. distance to the image tissue edge over the whole distance range (5-95% and 25-75% bands, mean), with the
rising piecewise-linear fit (vs. constant by AIC) over that whole range and its elbow, as panel C of the main figure.
The analysis itself fits only within 100 µm of the edge and tests the elbow against interior windows; this figure
shows what a single fit over the whole section picks up instead.

In [ ]:
from bosperrus import PiecewiseLinearFit
from bosperrus.anndata_api import fit_border_profile

GLOBAL_BLUE = json.loads(Path("../../fit_palette.json").read_text())["Piecewise Linear Fit"]


def sig2(v):
    """Two significant figures, written out (180, 3200, 13)."""
    return f"{float(f'{v:.2g}'):g}"


def global_fit_panel(ax, sample, transform=None):
    """Whole-distance rising piecewise-linear fit of the sample's largest image-mask component (of transform(counts)
    if given)."""
    bins = pd.read_parquet(
        RESULTS_DIR / f"{sample}_filtered_bins_imagemask.parquet",
        columns=["components", "virtual", "distance_to_border", "n_counts"],
    )
    bins = bins[(bins["components"] >= 0) & ~bins["virtual"]]
    label = bins["components"].value_counts().idxmax()
    comp = bins[bins["components"] == label]
    d, y = comp["distance_to_border"].to_numpy(), comp["n_counts"].to_numpy()
    if transform is not None:
        y = transform(y)
    fit = fit_border_profile(d, y, rising_only=True)
    p = fit["params"]
    if "piecewise_linear_b" in p:

        def predict(x):
            return PiecewiseLinearFit.piecewise_plateau(
                np.asarray(x, float), p["piecewise_linear_b"], p["piecewise_linear_m"], p["piecewise_linear_c"]
            )

    else:

        def predict(x):
            return np.full(np.shape(x), p["constant_c"], dtype=float)

    d_max = d.max()
    step = 8 if d_max <= 600 else 40
    plot_fit(
        ax,
        d,
        y,
        predict,
        bins=np.r_[np.arange(0, min(200, d_max), 8), np.arange(200, d_max + step, step)] if d_max > 200
        else np.arange(0, d_max + 8, 8),
        d_grid=np.linspace(0, d_max, 400),
        line_kwargs={"color": GLOBAL_BLUE, "linewidth": 1.3},
    )
    if fit["border_effect"]:
        knot = fit["elbow_um"]
        ax.axvspan(0, knot, color=GLOBAL_BLUE, alpha=0.12, lw=0)
        ax.axvline(knot, color=GLOBAL_BLUE, ls="--", lw=0.9)
        verdict = f"elbow {sig2(knot)} µm"
    else:
        verdict = "no rising elbow"
    ax.set_xlim(0, d_max)
    ax.set_ylim(bottom=0)
    area_mm2 = len(comp) * 64 / 1e6
    ax.set_title(f"{sample} ({sig2(area_mm2)} mm²)\n{verdict}", loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
    return {"sample": sample, "component": int(label), "area_mm2": area_mm2, "elbow_um": fit["elbow_um"],
            "half_life_um": fit["half_life_um"], "effect_strength": fit["observed_effect_strength"]}


with plt.rc_context({}):  # Arial 12 from the first cell
    fig, axes = plt.subplots(2, 4, figsize=(9, 4), layout="constrained")
    rows = [global_fit_panel(ax, s) for ax, s in zip(axes.flat, SAMPLE_ORDER, strict=True)]
    fig.supxlabel("distance to tissue edge (µm)", fontsize=12)
    fig.supylabel("counts per 8 µm bin", fontsize=12)
    plt.savefig("ST_global_fits_supp.svg", bbox_inches="tight", dpi=300)
    plt.show()

print(pd.DataFrame(rows).round(3).to_string(index=False))

# Supplement: global fits on log counts

As the previous supplement, with log(1 + counts per 8 µm bin) instead of counts: the same rising piecewise-linear fit
(vs. constant by AIC) over the whole distance range of each sample's largest component.

In [ ]:
with plt.rc_context({}):  # Arial 12 from the first cell
    fig, axes = plt.subplots(2, 4, figsize=(9, 4), layout="constrained")
    rows = [global_fit_panel(ax, s, transform=np.log1p) for ax, s in zip(axes.flat, SAMPLE_ORDER, strict=True)]
    fig.supxlabel("distance to tissue edge (µm)", fontsize=12)
    fig.supylabel("log(1 + counts per 8 µm bin)", fontsize=12)
    plt.savefig("ST_global_fits_log_supp.svg", bbox_inches="tight", dpi=300)
    plt.show()

print(pd.DataFrame(rows).round(3).to_string(index=False))